# Day 21 — Build Systems, Visualized

A build system topologically sorts a dependency graph (build prereqs first) and rebuilds
only what a change affects. Here's the graph and the incremental logic.

In [ ]:
DEPS = {
    'app': ['main.o', 'util.o'],
    'main.o': ['main.c', 'shared.h'],
    'util.o': ['util.c', 'shared.h'],
    'main.c': [], 'util.c': [], 'shared.h': [],
}

def build_order(deps):
    color = {n: 0 for n in set(deps) | {p for ps in deps.values() for p in ps}}
    order = []
    def visit(n):
        color[n] = 1
        for p in sorted(deps.get(n, [])):
            if color[p] == 0: visit(p)
        color[n] = 2; order.append(n)
    for n in sorted(color):
        if color[n] == 0: visit(n)
    return order

print('build order:', build_order(DEPS))

## Which change forces which rebuild?

In [ ]:
from collections import defaultdict

def rebuild_set(deps, changed):
    rev = defaultdict(set)
    for t, ps in deps.items():
        for p in ps: rev[p].add(t)
    out, stack = set(), list(changed)
    while stack:
        for dep in rev.get(stack.pop(), ()):
            if dep not in out: out.add(dep); stack.append(dep)
    return out

print("edit main.c  -> rebuild", rebuild_set(DEPS, {'main.c'}))
print("edit shared.h -> rebuild", rebuild_set(DEPS, {'shared.h'}))
print("edit nothing  -> rebuild", rebuild_set(DEPS, set()))

## Takeaways

- Topological sort orders the build so prerequisites come first (a cycle is an error).
- Incremental builds skip anything whose prerequisites didn't change — why `make` is fast.
- Editing a shared header cascades to everything downstream.

**Now build** build_order/needs_rebuild/rebuild_set in `homework.py`, then `pytest -q`.
openpilot's scons does exactly this at scale.